# Bitcoin Solo Mining Profitability Calculator


In [ ]:
# Install dependencies
!pip install -q requests pandas matplotlib numpy


# Bitcoin Solo Mining Profitability Calculator

This notebook computes expected profitability for solo Bitcoin mining, accounting for hardware specs, difficulty, electricity costs, and wallet transaction fees.

**Source:** Analysis inspired by Bitcoin wallet selection guides for solo miners.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# Mining parameters
hash_rate_th = 100  # Hash rate in TH/s (terahashes per second)
power_consumption_w = 3250  # Power consumption in watts
electricity_cost_kwh = 0.08  # Cost per kWh in USD

# Bitcoin network parameters (as of 2024)
current_difficulty = 84_000_000_000_000  # Example difficulty
block_reward_btc = 6.25  # Current block reward (post-halving)
btc_price_usd = 42000  # Current BTC price

# Wallet/mining parameters
transaction_fee_btc = 0.0001  # Transaction fee per payout
minimum_payout_btc = 0.01  # Minimum payout threshold

print(f"Mining Configuration:")
print(f"Hash Rate: {hash_rate_th} TH/s")
print(f"Power: {power_consumption_w} W")
print(f"Electricity Cost: ${electricity_cost_kwh}/kWh")
print(f"Difficulty: {current_difficulty}")
print(f"Block Reward: {block_reward_btc} BTC")
print(f"BTC Price: ${btc_price_usd}")

In [ ]:
# Calculate expected time to find a block
# Expected time (seconds) = difficulty * 2^32 / hash_rate
expected_time_seconds = (current_difficulty * (2**32)) / (hash_rate_th * 1e12)
expected_time_days = expected_time_seconds / (24 * 3600)
expected_time_years = expected_time_days / 365

print(f"\nExpected Time to Find Block:")
print(f"  {expected_time_days:.2f} days")
print(f"  {expected_time_years:.3f} years")
print(f"  {expected_time_seconds:.0f} seconds")

# Daily operating cost
power_consumption_kw = power_consumption_w / 1000
daily_power_usage_kwh = power_consumption_kw * 24
daily_electricity_cost = daily_power_usage_kwh * electricity_cost_kwh

print(f"\nDaily Operating Cost:")
print(f"  Daily Power Usage: {daily_power_usage_kwh:.2f} kWh")
print(f"  Daily Cost: ${daily_electricity_cost:.2f}")

In [ ]:
# Expected annual profitability (probabilistic)
blocks_per_year = 365 * 24 * 3600 / expected_time_seconds
btc_per_year = blocks_per_year * block_reward_btc
usd_revenue_per_year = btc_per_year * btc_price_usd

annual_electricity_cost = daily_electricity_cost * 365
annual_transaction_fees = (blocks_per_year * transaction_fee_btc * btc_price_usd)

net_profit_usd = usd_revenue_per_year - annual_electricity_cost - annual_transaction_fees
roi_percent = (net_profit_usd / (power_consumption_w * 365 * 24 / 1000)) * 100 if annual_electricity_cost > 0 else 0

print(f"\nAnnual Profitability Estimate:")
print(f"  Expected BTC/year: {btc_per_year:.6f}")
print(f"  Revenue (USD): ${usd_revenue_per_year:.2f}")
print(f"  Electricity Cost: ${annual_electricity_cost:.2f}")
print(f"  Transaction Fees: ${annual_transaction_fees:.2f}")
print(f"  Net Profit (USD): ${net_profit_usd:.2f}")
if net_profit_usd > 0:
    print(f"  Status: PROFITABLE")
else:
    print(f"  Status: NOT PROFITABLE")

In [ ]:
# Scenario analysis: vary hash rate and electricity cost
hash_rates = np.array([10, 50, 100, 150, 200])  # TH/s
electricity_costs = np.array([0.05, 0.08, 0.10, 0.15])  # $/kWh

profitability_matrix = np.zeros((len(hash_rates), len(electricity_costs)))

for i, hr in enumerate(hash_rates):
    for j, ec in enumerate(electricity_costs):
        time_to_block = (current_difficulty * (2**32)) / (hr * 1e12)
        blocks_yr = 365 * 24 * 3600 / time_to_block
        btc_yr = blocks_yr * block_reward_btc
        revenue = btc_yr * btc_price_usd
        
        daily_cost = (power_consumption_w / 1000) * 24 * ec
        annual_cost = daily_cost * 365
        tx_fees = blocks_yr * transaction_fee_btc * btc_price_usd
        
        profit = revenue - annual_cost - tx_fees
        profitability_matrix[i, j] = profit

print("\nProfitability Matrix (Annual USD Profit):")
print("Hash Rate (TH/s) vs Electricity Cost ($/kWh)\n")
print("Hash Rate\t", end="")
for ec in electricity_costs:
    print(f"${ec}/kWh\t", end="")
print()
for i, hr in enumerate(hash_rates):
    print(f"{hr} TH/s\t\t", end="")
    for j in range(len(electricity_costs)):
        print(f"${profitability_matrix[i, j]:.0f}\t", end="")
    print()

In [ ]:
# Visualize profitability scenarios
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
im = axes[0].imshow(profitability_matrix, cmap='RdYlGn', aspect='auto')
axes[0].set_xticks(range(len(electricity_costs)))
axes[0].set_yticks(range(len(hash_rates)))
axes[0].set_xticklabels([f'${ec}/kWh' for ec in electricity_costs])
axes[0].set_yticklabels([f'{hr} TH/s' for hr in hash_rates])
axes[0].set_xlabel('Electricity Cost')
axes[0].set_ylabel('Hash Rate')
axes[0].set_title('Annual Profitability (USD)')
fig.colorbar(im, ax=axes[0])

for i in range(len(hash_rates)):
    for j in range(len(electricity_costs)):
        text = axes[0].text(j, i, f'${profitability_matrix[i, j]:.0f}',
                          ha="center", va="center", color="black", fontsize=9)

# Line plot: impact of electricity cost on profitability
for hr in [50, 100, 150]:
    profits = []
    for ec in electricity_costs:
        time_to_block = (current_difficulty * (2**32)) / (hr * 1e12)
        blocks_yr = 365 * 24 * 3600 / time_to_block
        btc_yr = blocks_yr * block_reward_btc
        revenue = btc_yr * btc_price_usd
        daily_cost = (power_consumption_w / 1000) * 24 * ec
        annual_cost = daily_cost * 365
        tx_fees = blocks_yr * transaction_fee_btc * btc_price_usd
        profit = revenue - annual_cost - tx_fees
        profits.append(profit)
    axes[1].plot(electricity_costs, profits, marker='o', label=f'{hr} TH/s')

axes[1].axhline(y=0, color='r', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Electricity Cost ($/kWh)')
axes[1].set_ylabel('Annual Profit (USD)')
axes[1].set_title('Profitability vs Electricity Cost')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nVisualization complete. Higher hash rates and lower electricity costs yield better profitability.")

## Key Insights for Wallet Selection

- **Payout Efficiency**: Wallets with lower minimum payout thresholds reduce idle capital and fees.
- **Break-Even Analysis**: If annual profit is negative, even the best wallet cannot save an unprofitable operation.
- **Fee Impact**: Transaction fees (shown above) reduce net profit; choose wallets with fair fee structures.
- **Hardware Investment**: The profitability matrix shows that hash rate dominance matters far more than wallet features.

Use this calculator to evaluate whether solo mining is viable for your setup before selecting a wallet.

---

## Reference

[see the full tutorial](https://protraderdaily.com/crypto/bitcoin-wallet-for-solo-mining)
